### Currency Conversion Tool

If we ask a normal LLM to convert currency then it can answer but it will be based on the period it was trained on. If we want to have it convert in realtime rates, we will need to call a tool. We will use ExchangeRate-API for this. (Now LLMs can tell the realtime rates since they have access to websearch)

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv
import json

In [2]:
load_dotenv()

True

### Tool Create

In [3]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/9929acb59fbc45fd6b9327ce/pair/{base_currency}/{target_currency}"
    
    response = requests.get(url)
    
    print("CONVERSION FACTOR TOOL CALLED")

    return response.json()

In [4]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

CONVERSION FACTOR TOOL CALLED


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1790294401,
 'time_last_update_utc': 'Fri, 25 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1790380801,
 'time_next_update_utc': 'Sat, 26 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.0151}

In [5]:
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    print("CONVERT TOOL CALLED")

    return base_currency_value * conversion_rate

In [6]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [7]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

CONVERT TOOL CALLED


851.5999999999999

### Tool Binding

In [8]:
llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')

In [9]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])   # bound 2 tools to an llm

In [10]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [11]:
# ai_message = llm_with_tools.invoke(messages)
# messages.append(ai_message)
# ai_message.tool_calls

In [12]:
# for tool_call in ai_message.tool_calls:
#   # execute the 1st tool and get the value of conversion rate
#   if tool_call['name'] == 'get_conversion_factor':
#     tool_message1 = get_conversion_factor.invoke(tool_call)
    
#     # fetch this conversion rate
#     conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    
#     # append this tool message to messages list
#     messages.append(tool_message1)
    
#   # execute the 2nd tool using the conversion rate from tool 1
#   if tool_call['name'] == 'convert':
      
#     # fetch the current arg
#     tool_call['args']['conversion_rate'] = conversion_rate
#     tool_message2 = convert.invoke(tool_call)
#     messages.append(tool_message2)

In [13]:
while True:
    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)

    if not ai_message.tool_calls:
        break

    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "get_conversion_factor":
            tool_message = get_conversion_factor.invoke(tool_call)

            conversion_rate = json.loads(
                tool_message.content
            )["conversion_rate"]

            messages.append(tool_message)

        elif tool_call["name"] == "convert":
            tool_call["args"]["conversion_rate"] = conversion_rate

            tool_message = convert.invoke(tool_call)

            messages.append(tool_message)

CONVERSION FACTOR TOOL CALLED
CONVERT TOOL CALLED


In [14]:
print(ai_message.text)

The current conversion factor from INR to USD is 0.01042. Based on this rate, 10 INR is equal to 0.1042 USD.
